# Multi-Goal Financial Asset Recommender System

**RL-Driven Transformer Pipeline:**
- **[PHASE 1 — Member A] Autoencoder Embeddings**: Trains a masked weighted autoencoder on ALL historical data with exponential time-decay weighting. Produces compressed 8-dimensional behavioral embeddings per asset.
- **[PHASE 2–3 — Member D] RL Transformer + Gaussian Policy**: A Transformer processes the entire market universe via Self-Attention. A Gaussian Policy head outputs (μ, σ) per asset — target allocation and exploration noise.
- **[PHASE 4–5 — Member D + C] Simulator Environment + REINFORCE**: Sampled weights are passed into the non-differentiable simulator. An inverse-exponential reward is returned. The Policy Gradient theorem updates the Transformer without needing gradients through the simulator.

### Architecture
All function definitions live in worker modules (`_*.py`). This notebook is a **thin orchestration layer** — edit `_constants.py` for config, dive into workers for implementation details.

In [4]:
%pip install yfinance matplotlib seaborn scipy lxml html5lib requests tqdm nbformat joblib
# Install CUDA-enabled PyTorch for GPU acceleration
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install pandas-datareader

from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
import pandas_datareader
warnings.filterwarnings('ignore')

plt.style.use('default')
%matplotlib inline

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached pandas_datareader-0.10.0-py3-none-any.whl.metadata (2.9 kB)
Using cached pandas_datareader-0.10.0-py3-none-any.whl (109 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Phase 1 — Member A: Data Scraping & Autoencoder Embeddings

In [17]:
import importlib
import _data_worker

importlib.reload(_data_worker)

<module '_data_worker' from 'c:\\SJSU\\CMPE 256\\Financial_Analytics\\_data_worker.py'>

In [18]:
# ── Configure Pipeline ──
from _constants import DEFAULT_PIPELINE_CONFIG, DataSyncMode
config = DEFAULT_PIPELINE_CONFIG.copy()
#config["data_source_mode"] = DataSyncMode.OFFLINE_CSV_ONLY  # Override for this run
config["data_source_mode"] = DataSyncMode.FULL_REBUILD  # Override for this run

# ── Fetch Universe & Build Dataset ──
from _data_worker import fetch_macro_universe, generate_dataset_member_a, run_data_diagnostics

tickers = fetch_macro_universe() if config["data_source_mode"] != DataSyncMode.OFFLINE_CSV_ONLY else []
master_df, price_matrix, volume_matrix, daily_returns, drip_returns, annual_inflation, inflation_daily_returns, inflation_index  = generate_dataset_member_a(tickers, config)
print(master_df.columns[master_df.columns.duplicated()])
# ── Validate Data Quality ──
run_data_diagnostics(master_df, config)

[05:01:27] [Member A] Fetching latest S&P 1500 constituents from Wikipedia...
[05:01:28] [Member A] Fetching complete US ETF universe (~3500+ symbols) via Nasdaq FTP...
[05:01:32] [Member A] Discovered 5,193 active ETF symbols.
[05:01:32] [Member A] Total combined macro universe: 6,699 symbols.
[05:01:53] [Member A] Booting in mode [FULL_REBUILD]
[05:01:53] [Member A] Pulling full history for 9 new tickers from 1962-01-01...


New Tickers:   0%|          | 0/1 [00:00<?, ?it/s]

$WDIG: possibly delisted; no timezone found
$WDAI: possibly delisted; no timezone found
$XNDX: possibly delisted; no timezone found
$BUYB: possibly delisted; no timezone found
$JUDB: possibly delisted; no timezone found
$PSAI: possibly delisted; no price data found  (1d 1962-01-01 -> 2026-05-07)
$OCDB: possibly delisted; no timezone found
$APDB: possibly delisted; no timezone found
$KMCA: possibly delisted; no timezone found

9 Failed downloads:
['WDIG', 'WDAI', 'XNDX', 'BUYB', 'JUDB', 'OCDB', 'APDB', 'KMCA']: possibly delisted; no timezone found
['PSAI']: possibly delisted; no price data found  (1d 1962-01-01 -> 2026-05-07)


[05:01:53] [Member A] Fetching Info/Fundamentals for 6699 tickers...


Fundamentals:   0%|          | 0/6699 [00:00<?, ?it/s]

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BUYB"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WDAI"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WDIG"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: XNDX"}}}


[06:00:35] [Member A] WARNING: 6617 fundamental conflicts. Last write wins applied.

[06:02:34] [Member A] Preprocessing Configured ML Features...
   -> Extracting Categorical Matrix: 'sector'
   -> Extracting Categorical Matrix: 'industry'
   -> Extracting Categorical Matrix: 'state'
   -> Extracting Categorical Matrix: 'quoteType'
   -> Extracting Categorical Matrix: 'exchange'

=== DATAFRAME DIAGNOSTICS & SYSTEM MAPPING ===
Total Equities Tracked: 6645
Total Columns Processed per Asset: 847

All Features Extracted in Master DF:
['phone', 'longBusinessSummary', 'companyOfficers', 'executiveTeam', 'maxAge', 'priceHint', 'previousClose', 'open', 'dayLow', 'dayHigh', 'regularMarketPreviousClose', 'regularMarketOpen', 'regularMarketDayLow', 'regularMarketDayHigh', 'trailingPE', 'volume', 'regularMarketVolume', 'averageVolume', 'averageVolume10days', 'averageDailyVolume10Day', 'bid', 'ask', 'bidSize', 'askSize', 'yield', 'totalAssets', 'fiftyTwoWeekLow', 'fiftyTwoWeekHigh', 'allTimeHigh',

In [19]:
# ── Phase 1 Persistence Orchestration ──
from _ml_worker import train_pytorch_embedding_model, save_embedding_cache, load_embedding_cache

# 1. Try to load from disk first using config settings
DATA_CACHE = load_embedding_cache(
    master_df, price_matrix, volume_matrix, daily_returns, 
    config, drip_daily_returns=drip_returns, folder=config["ml_cache_dir"]
)

# 2. Train only if cache is missing or force-retrain is ON
if DATA_CACHE is None or config["ml_force_retrain"]:
    print(f"[{time.strftime('%H:%M:%S')}] No valid cache found. Starting training...")
    DATA_CACHE = train_pytorch_embedding_model(
        master_df, price_matrix, volume_matrix, daily_returns,
        config, drip_daily_returns=drip_returns
    )
    save_embedding_cache(DATA_CACHE, folder=config["ml_cache_dir"])
else:
    print(f"[{time.strftime('%H:%M:%S')}] SUCCESS: Loaded pre-trained embeddings from '{config['ml_cache_dir']}/'")

# 3. Generate Walk-Forward Snapshots if enabled (or force rebuild is ON)
from _ml_worker import generate_walkforward_embeddings_monthly
if config.get("wf_enabled", False) or config.get("wf_force_rebuild", False):
    print(f"[{time.strftime('%H:%M:%S')}] Checking/Building Walk-Forward Cache...")
    generate_walkforward_embeddings_monthly(DATA_CACHE["model_checkpoint"], ticker_data, config, None, None, start_year=config.get("wf_start_year", 2000), force_rebuild=config.get("wf_force_rebuild", False))


[06:05:40] [Member A] Persistence: Found existing model for architecture ed8_dm64_lr001. Loading...
[06:05:40] SUCCESS: Loaded pre-trained embeddings from 'cache/'


## Phases 2–5 — Member D: End-to-End RL Transformer Portfolio Optimizer

**Replaces the old Member B/C scoring pipeline** with an End-to-End Reinforcement Learning system.

In [20]:
# ── Universal Policy RL Training (Member D) ──
import importlib
import _rl_worker, _sim_worker, _constants
importlib.reload(_constants)
importlib.reload(_sim_worker)
importlib.reload(_rl_worker)
from _rl_worker import train_rl_agent
from _constants import TEST_PROFILES
import pandas as pd
from IPython.display import display

print(f"Loaded {len(TEST_PROFILES)} User Profiles for Universal Training")
print(f"Risk Range: {min(p['risk_tolerance'] for p in TEST_PROFILES)} - {max(p['risk_tolerance'] for p in TEST_PROFILES)}")
print(f"Capital Range: ${min(p['start_cap'] for p in TEST_PROFILES):,.0f} - ${max(p['start_cap'] for p in TEST_PROFILES):,.0f}")

# Run the Universal Policy training (agent randomly samples from ALL profiles internally)
# user_profile here is just used for the FINAL greedy evaluation output
rl_results = train_rl_agent(DATA_CACHE, TEST_PROFILES[0], config, verbose=True)

final_portfolio = rl_results.get('portfolio_weights', {})
history = rl_results.get('training_history', [])

print(f"\n{'=' * 60}")
print(f"  UNIVERSAL AGENT TRAINING COMPLETE")
print(f"  Selected Assets: {len(final_portfolio)}")
print(f"{'=' * 60}")

if final_portfolio:
    print(f"\n  {'Ticker':<10} {'Weight':<10}")
    print(f"  {'-'*25}")
    for ticker, weight in sorted(final_portfolio.items(), key=lambda x: x[1], reverse=True)[:20]:
        print(f"  {ticker:<10} {weight:<10.2%}")

# Plot reward convergence
if history:
    rewards = [h['reward'] for h in history]
    plt.figure(figsize=(12, 4))
    plt.plot(rewards, alpha=0.3, label='Per-Episode')
    # Rolling average
    window = min(20, len(rewards))
    if window > 1:
        rolling = pd.Series(rewards).rolling(window).mean()
        plt.plot(rolling, linewidth=2, label=f'Rolling {window}-ep Mean')
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.title('Universal Policy RL Agent — Reward Convergence')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


Loaded 30 User Profiles for Universal Training
Risk Range: 1.0 - 9.0
Capital Range: $2,500 - $2,000,000
[06:06:06] [Member D] Saving shared data to cache\_rl_shared_data.pkl...
[06:06:07] [Member D] Launching 3 independent agent processes...
  -> Logs: c:\SJSU\CMPE 256\Financial_Analytics\logs/agent_{0..2}.log
  -> Monitor with: Get-Content logs/agent_0.log -Wait
  -> Agent 0 started (PID 180964)
  -> Agent 1 started (PID 152240)
  -> Agent 2 started (PID 173448)
[06:08:08] [Member D] All 3 agents finished.

  UNIVERSAL AGENT TRAINING COMPLETE
  Selected Assets: 6539

  Ticker     Weight    
  -------------------------
  IRET       0.02%     
  EATZ       0.02%     
  DBSC       0.02%     
  SIXG       0.02%     
  CTRA       0.02%     
  QUVU       0.02%     
  TOKE       0.02%     
  GDEN       0.02%     
  SNPX       0.02%     
  LAYS       0.02%     
  MYMJ       0.02%     
  ZTAX       0.02%     
  MYMI       0.02%     
  ENPH       0.02%     
  XXX        0.02%     
  BUFE       

In [22]:
# ── Reward Convergence Plot ──
if 'history' in locals() and history:
    episodes = [h['episode'] for h in history]
    rewards  = [h['reward'] for h in history]
    gfrs     = [h['GFR'] for h in history]
    etvs     = [h['ETV'] for h in history]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(episodes, rewards, color='steelblue')
    axes[0].set_title('Reward')
    axes[1].plot(episodes, gfrs, color='seagreen')
    axes[1].set_title('GFR')
    axes[2].plot(episodes, etvs, color='darkorange')
    axes[2].set_title('ETV')
    plt.show()